In [7]:
# Setup
import duckdb
import pandas as pd
from pathlib import Path

# Resolve the database path relative to the ingestion/ folder
DB_PATH = str(Path("../raw_data/my_cost_of_living.duckdb").resolve())
print(DB_PATH)

C:\Projects\my-cost-of-living\raw_data\my_cost_of_living.duckdb


In [8]:
# Load source files into the raw schema (incremental)
FORCE_RELOAD = False  # True reloads every month

months = pd.date_range(start="2022-01", end=pd.Timestamp.today(), freq="MS").strftime("%Y_%m").tolist()

sources = {
    f"pricecatcher_{m}": f"https://storage.data.gov.my/pricecatcher/pricecatcher_{m.replace('_', '-')}.parquet"
    for m in months
}
sources.update({
    "lookup_premise": "https://storage.data.gov.my/pricecatcher/lookup_premise.parquet",
    "lookup_item": "https://storage.data.gov.my/pricecatcher/lookup_item.parquet",
    "fuelprice": "https://storage.data.gov.my/commodities/fuelprice.parquet",
})

con = duckdb.connect(DB_PATH)
try:
    con.execute("create schema if not exists raw")

    existing = {
        r[0] for r in con.execute(
            "select table_name from information_schema.tables where table_schema = 'raw'"
        ).fetchall()
    }
    loaded_months = sorted(t for t in existing if t.startswith("pricecatcher_"))
    latest_loaded = loaded_months[-1] if loaded_months else None

    for table_name, url in sources.items():
        is_monthly = table_name.startswith("pricecatcher_")

        # Skip closed months that are already loaded. Always reload the latest
        # loaded month and the small snapshot tables.
        if not FORCE_RELOAD and is_monthly and table_name in existing and table_name != latest_loaded:
            print(f"{table_name}: skipped (already loaded)")
            continue

        try:
            # Idempotent load with lineage metadata: load time and source file
            con.execute(
                f"""
                create or replace table raw.{table_name} as
                select
                    *,
                    current_timestamp as _loaded_at,
                    '{url}' as _source_url
                from read_parquet('{url}')
                """
            )
        except duckdb.Error as e:
            
            if is_monthly and "404" in str(e):
                print(f"{table_name}: not published yet (404), skipped")
                continue
            raise

        row_count = con.execute(f"select count(*) from raw.{table_name}").fetchone()[0]
        print(f"{table_name}: {row_count} rows")
finally:
    con.close()

pricecatcher_2022_01: skipped (already loaded)
pricecatcher_2022_02: skipped (already loaded)
pricecatcher_2022_03: skipped (already loaded)
pricecatcher_2022_04: skipped (already loaded)
pricecatcher_2022_05: skipped (already loaded)
pricecatcher_2022_06: skipped (already loaded)
pricecatcher_2022_07: skipped (already loaded)
pricecatcher_2022_08: skipped (already loaded)
pricecatcher_2022_09: skipped (already loaded)
pricecatcher_2022_10: skipped (already loaded)
pricecatcher_2022_11: skipped (already loaded)
pricecatcher_2022_12: skipped (already loaded)
pricecatcher_2023_01: skipped (already loaded)
pricecatcher_2023_02: skipped (already loaded)
pricecatcher_2023_03: skipped (already loaded)
pricecatcher_2023_04: skipped (already loaded)
pricecatcher_2023_05: skipped (already loaded)
pricecatcher_2023_06: skipped (already loaded)
pricecatcher_2023_07: skipped (already loaded)
pricecatcher_2023_08: skipped (already loaded)
pricecatcher_2023_09: skipped (already loaded)
pricecatcher_

In [9]:
# Inspect loaded tables (read-only)
con = duckdb.connect(DB_PATH, read_only=True)
tables = con.sql("show all tables").df()
con.close()
tables

,database,schema,name,column_names,column_types,temporary
0,my_cost_of_living,main,stg_fuelprice,"[series_type, effective_date, ron95, ron97, di...","[VARCHAR, DATE, DOUBLE, DOUBLE, DOUBLE, DOUBLE...",False
1,my_cost_of_living,main,stg_lookup_item,"[item_code, item_name, unit, item_group, item_...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, T...",False
2,my_cost_of_living,main,stg_lookup_premise,"[premise_code, premise_name, address, premise_...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
3,my_cost_of_living,main,stg_pricecatcher,"[price_date, premise_code, item_code, price, _...","[TIMESTAMP, BIGINT, BIGINT, DOUBLE, TIMESTAMP ...",False
4,my_cost_of_living,main,stg_pricecatcher_2026_10,"[price_date, premise_code, item_code, price, _...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False
...,...,...,...,...,...,...
62,my_cost_of_living,raw,pricecatcher_2026_07,"[date, premise_code, item_code, price, _loaded...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False
63,my_cost_of_living,raw,pricecatcher_2026_08,"[date, premise_code, item_code, price, _loaded...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False
64,my_cost_of_living,raw,pricecatcher_2026_09,"[date, premise_code, item_code, price, _loaded...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False
65,my_cost_of_living,raw,pricecatcher_2026_10,"[date, premise_code, item_code, price, _loaded...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False


In [10]:
# Quick peek (read-only)
con = duckdb.connect(DB_PATH, read_only=True)
df = con.sql("select * from main.stg_pricecatcher limit 10").df()
con.close()
df

,price_date,premise_code,item_code,price,_loaded_at,_source_url
0,2022-01-01,2,1,9.1,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
1,2022-01-01,2,9,36.0,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
2,2022-01-01,2,14,24.0,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
3,2022-01-01,2,16,4.3,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
4,2022-01-01,2,18,4.5,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
5,2022-01-01,2,19,4.5,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
6,2022-01-01,2,21,3.5,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
7,2022-01-01,2,22,10.0,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
8,2022-01-01,2,24,5.0,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...
9,2022-01-01,2,25,2.8,2026-11-07 08:20:27.534631+08:00,https://storage.data.gov.my/pricecatcher/price...


In [11]:
def peek(query):
    # Run a read-only query and release the file straight away
    con = duckdb.connect(DB_PATH, read_only=True)
    df = con.sql(query).df()
    con.close()
    return df


# which item codes are missing, and when they were seen
peek("""
select p.item_code, count(*) as n_rows,
       min(p.price_date) as first_seen, max(p.price_date) as last_seen
from main.stg_pricecatcher p
left join main.stg_lookup_item i on p.item_code = i.item_code
where i.item_code is null and p.item_code != -1
group by 1 order by n_rows desc
""")

# same for premises
peek("""
select p.premise_code, count(*) as n_rows,
       min(p.price_date) as first_seen, max(p.price_date) as last_seen
from main.stg_pricecatcher p
left join main.stg_lookup_premise l on p.premise_code = l.premise_code
where l.premise_code is null and p.premise_code != -1
group by 1 order by n_rows desc
""")

# the premise with a null state
peek("select * from main.stg_lookup_premise where state is null")

,premise_code,premise_name,address,premise_type,state,district,_loaded_at,_source_url
0,-1,Unknown Premise,Unknown Address,None,None,None,2026-11-07 10:51:06.199838+08:00,https://storage.data.gov.my/pricecatcher/looku...


In [12]:
peek("select * from main.stg_lookup_item where item_code = -1")

,item_code,item_name,unit,item_group,item_category,_loaded_at,_source_url
0,-1,None,None,None,None,2026-11-07 10:51:06.624004+08:00,https://storage.data.gov.my/pricecatcher/looku...


In [13]:
peek("""
select p.item_code, count(*) as n_rows,
       min(p.price_date) as first_seen, max(p.price_date) as last_seen
from main.stg_pricecatcher p
left join main.stg_lookup_item i on p.item_code = i.item_code
where i.item_code is null and p.item_code != -1
group by 1 order by n_rows desc
""")

,item_code,n_rows,first_seen,last_seen
0,2065,117445,2024-04-01,2025-11-03
1,2064,103370,2024-04-01,2025-11-03
2,2070,102611,2024-04-01,2025-11-03
3,2071,102602,2024-04-01,2025-08-25
4,2072,82044,2024-04-01,2025-08-25
5,2057,62900,2023-12-31,2026-10-26
6,2043,57635,2024-01-01,2025-07-14
7,2024,56648,2024-01-01,2025-07-14
8,2033,56573,2024-01-01,2025-07-14
9,2023,55077,2024-01-01,2026-10-26


In [14]:
peek("""
select p.premise_code, count(*) as n_rows,
       min(p.price_date) as first_seen, max(p.price_date) as last_seen
from main.stg_pricecatcher p
left join main.stg_lookup_premise l on p.premise_code = l.premise_code
where l.premise_code is null and p.premise_code != -1
group by 1 order by n_rows desc
""")

,premise_code,n_rows,first_seen,last_seen
0,20927,273,2024-02-05,2024-02-19


In [15]:
peek("select min(item_code), max(item_code), count(*) from main.stg_lookup_item")

,min(item_code),max(item_code),count_star()
0,-1,2115,797


In [16]:
peek("""
select item_code, item_name, item_group, item_category
from main.stg_lookup_item
where item_code between 2000 and 2115
order by item_code
""")

,item_code,item_name,item_group,item_category
0,2000,PENCUCI PELBAGAI GUNA AJAX FABULOSO (LAVENDER),PRODUK KEBERSIHAN,PENJAGAAN RUMAH
1,2001,DUMEX DUPRO 2 (RUMUSAN SUSULAN),SUSU DAN BARANGAN BAYI,SUSU BAYI
2,2002,PELUNTUR CLOROX (ORIGINAL),PRODUK KEBERSIHAN,PENJAGAAN RUMAH
3,2003,BIHUN KERING IMPORT (PELBAGAI JENAMA),BARANGAN BERBUNGKUS,BIHUN
4,2004,BERAS SUPER IMPORT CAP FAMILY,BARANGAN BERBUNGKUS,BERAS
5,2008,SUSU TEPUNG PENUH KRIM SEGERA NESPRAY,BARANGAN BERBUNGKUS,KRIMER DAN SUSU TEPUNG
6,2009,DUTCH LADY 123 (PELBAGAI PERISA),SUSU DAN BARANGAN BAYI,SUSU BAYI
7,2010,DUTCH LADY 456 (PELBAGAI PERISA),SUSU DAN BARANGAN BAYI,SUSU BAYI
8,2011,KIUB SUP TOM YAM (MAGGI),BARANGAN BERBUNGKUS,REMPAH RATUS (BERBUNGKUS)
9,2012,KICAP TAMIN (DARK SOY SAUCE),BARANGAN BERBUNGKUS,KICAP DAN SOS


In [17]:
months = pd.date_range(start="2022-01", end="2026-10", freq="MS").strftime("%Y_%m").tolist()
raw_sql = " union all ".join(f"select count(*) as n from raw.pricecatcher_{m}" for m in months)
raw_total = peek(f"select sum(n) from ({raw_sql})").iloc[0, 0]
stg_total = peek("select count(*) from main.stg_pricecatcher").iloc[0, 0]
print(raw_total, stg_total, raw_total == stg_total)

129028914.0 129028914 True


In [18]:
peek("select min(date), max(date), count(*) from raw.pricecatcher_2026_10")

,min(date),max(date),count_star()
0,2026-10-10,2026-10-31,1359706


In [19]:
peek("""
select strftime(price_date, '%Y-%m') as month,
       count(distinct price_date) as days_with_data,
       day(last_day(min(price_date))) as days_in_month,
       min(price_date) as first_date,
       max(price_date) as last_date
from main.stg_pricecatcher
group by 1
having count(distinct price_date) < day(last_day(min(price_date)))
order by 1
""")

,month,days_with_data,days_in_month,first_date,last_date
0,2022-01,30,31,2022-01-01,2022-01-31
1,2022-02,27,28,2022-02-01,2022-02-28
2,2022-05,29,31,2022-05-01,2022-05-31
3,2022-06,29,30,2022-06-01,2022-06-30
4,2022-07,30,31,2022-07-01,2022-07-31
5,2022-08,30,31,2022-08-01,2022-08-30
6,2022-11,28,30,2022-11-01,2022-11-30
7,2022-12,30,31,2022-12-01,2022-12-31
8,2023-01,29,31,2023-01-01,2023-01-31
9,2023-02,27,28,2023-02-01,2023-02-28


In [21]:
peek("select count(*) from information_schema.tables where table_schema = 'raw'")

,count_star()
0,62


In [22]:
peek("""
select count(*) as dup_groups, sum(n) as dup_rows from (
    select price_date, premise_code, item_code, count(*) as n
    from main.stg_pricecatcher
    group by 1, 2, 3
    having count(*) > 1
)
""")

,dup_groups,dup_rows
0,20613,41226.0


In [24]:
peek("select min(date), max(date), count(*) from raw.pricecatcher_2026_11")

,min(date),max(date),count_star()
0,2026-11-01,2026-11-06,547044


In [25]:
peek("""
select
    count(*) as dup_groups,
    count(*) filter (where n_prices = 1) as same_price,
    count(*) filter (where n_prices > 1) as different_price,
    count(*) filter (where n_files > 1) as across_two_files
from (
    select price_date, premise_code, item_code,
           count(distinct price) as n_prices,
           count(distinct _source_url) as n_files
    from main.stg_pricecatcher
    group by 1, 2, 3
    having count(*) > 1
)
""")

,dup_groups,same_price,different_price,across_two_files
0,20613,20613,0,0
